# 18 — Ablation Jumlah Fitur SFM
**Pertanyaan (Reviewer P6):** berapa banyak fitur SFM yang benar-benar dibutuhkan?
Apakah 9 fitur Model A berlebihan, atau tiap fitur berkontribusi?

**Rancangan (jujur, angka nyata):**
1. Urutkan 9 fitur SFM menurut *feature importance* dari model **joint** (CIC+UNSW),
   supaya subset kecil = fitur paling informatif (bukan urutan arbitrer).
2. Untuk tiap ukuran subset k in {3,5,7,9}, latih XGBoost dan ukur **MCC** pada:
   - **in-domain CIC** (latih+uji CIC), **in-domain UNSW**, dan **joint** (gabung, uji tiap test);
   - dua arah **cross-network** (CIC->UNSW, UNSW->CIC) sebagai konteks.
3. Simpan `feature_ablation.json` + kurva PNG, upload ke S3.

Semua protokol identik notebook 06 (Model A, biner, z-score per dataset, XGBoost sama,
`SEED=42`, tanpa kebocoran: scaler fit di train saja).
> Jalankan di **SageMaker** (butuh `cleaned_100.pkl` + CSV UNSW).

In [ ]:
# --- Bootstrap (identik nb06) ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('matplotlib','matplotlib')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import pickle, os, json, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score
from xgboost import XGBClassifier
CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../feature_ablation.json'; OUT_PNG='feature_ablation.png'
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); S3_PREFIX='unsw-far'
SEED=42; SUBSETS=[3,5,7,9]
print('CIC:', os.path.exists(CIC_PKL), '| UNSW tr:', os.path.exists(UNSW_TRAIN), '| te:', os.path.exists(UNSW_TEST))
print('=== SEL 1 SELESAI ===')

In [ ]:
# --- Mapping & util (identik nb06) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())
def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan).fillna(0.0)
    return out.astype(float).values
def make_xgb():
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=SEED,tree_method='hist')
def mcc_of(Xtr,ytr,Xte,yte,cols):
    m=make_xgb(); m.fit(Xtr[:,cols],ytr); return float(matthews_corrcoef(yte,m.predict(Xte[:,cols])))
print('=== SEL 2 SELESAI ===')

In [ ]:
# --- Muat data + z-score per dataset (identik nb06) ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
scc=StandardScaler().fit(Xc_tr_raw); Xc_tr=scc.transform(Xc_tr_raw); Xc_te=scc.transform(Xc_te_raw)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
scu=StandardScaler().fit(Xu_tr_raw); Xu_tr=scu.transform(Xu_tr_raw); Xu_te=scu.transform(Xu_te_raw)
print('CIC tr/te:',Xc_tr.shape,Xc_te.shape,'| UNSW tr/te:',Xu_tr.shape,Xu_te.shape)
print('=== SEL 3 SELESAI ===')

In [ ]:
# --- Ranking fitur via importance model JOINT (data-driven) ---
Xj_tr=np.vstack([Xc_tr,Xu_tr]); yj_tr=np.concatenate([yc_tr,y_utr])
mj=make_xgb(); mj.fit(Xj_tr,yj_tr)
imp=mj.feature_importances_
order=list(np.argsort(imp)[::-1])   # indeks fitur, terpenting dulu
ranked=[CANON[i] for i in order]
print('Ranking fitur (penting -> kurang):')
for r,i in enumerate(order,1): print(f'  {r}. {CANON[i]:10s} imp={imp[i]:.4f}')
print('=== SEL 4 SELESAI ===')

In [ ]:
# --- Ablation: MCC vs jumlah fitur (top-k dari ranking) ---
rows=[]
for k in SUBSETS:
    cols=order[:k]
    names=[CANON[i] for i in cols]
    mcc_cic  = mcc_of(Xc_tr,yc_tr, Xc_te,yc_te, cols)              # in-domain CIC
    mcc_unsw = mcc_of(Xu_tr,y_utr, Xu_te,y_ute, cols)             # in-domain UNSW
    mcc_joint_c = mcc_of(np.vstack([Xc_tr,Xu_tr]),np.concatenate([yc_tr,y_utr]), Xc_te,yc_te, cols)
    mcc_joint_u = mcc_of(np.vstack([Xc_tr,Xu_tr]),np.concatenate([yc_tr,y_utr]), Xu_te,y_ute, cols)
    mcc_c2u  = mcc_of(Xc_tr,yc_tr, Xu_te,y_ute, cols)             # cross CIC->UNSW
    mcc_u2c  = mcc_of(Xu_tr,y_utr, Xc_te,yc_te, cols)             # cross UNSW->CIC
    rows.append(dict(k=k, features=names, mcc_indomain_cic=round(mcc_cic,4),
                     mcc_indomain_unsw=round(mcc_unsw,4),
                     mcc_joint_cic=round(mcc_joint_c,4), mcc_joint_unsw=round(mcc_joint_u,4),
                     mcc_cross_cic2unsw=round(mcc_c2u,4), mcc_cross_unsw2cic=round(mcc_u2c,4)))
    print(f'k={k}: in-CIC={mcc_cic:.3f} in-UNSW={mcc_unsw:.3f} '
          f'joint(C/U)={mcc_joint_c:.3f}/{mcc_joint_u:.3f} '
          f'cross(C2U/U2C)={mcc_c2u:+.3f}/{mcc_u2c:+.3f}')
print('=== SEL 5 SELESAI ===')

In [ ]:
# --- Tabel + kurva + simpan + upload S3 ---
tab=pd.DataFrame(rows)
import IPython.display as ipd; ipd.display(tab[['k','mcc_indomain_cic','mcc_indomain_unsw','mcc_joint_cic','mcc_joint_unsw','mcc_cross_cic2unsw','mcc_cross_unsw2cic']])
fig,ax=plt.subplots(figsize=(7.2,4.4))
ks=[r['k'] for r in rows]
for key,lab,col in [('mcc_indomain_cic','in-domain CIC','#4C72B0'),
                    ('mcc_indomain_unsw','in-domain UNSW','#DD8452'),
                    ('mcc_joint_unsw','joint (uji UNSW)','#55A868'),
                    ('mcc_joint_cic','joint (uji CIC)','#8172B3')]:
    ax.plot(ks,[r[key] for r in rows],'o-',lw=2,color=col,label=lab)
ax.set_xlabel('Jumlah fitur SFM (top-k menurut importance joint)')
ax.set_ylabel('MCC'); ax.set_xticks(ks); ax.set_ylim(0,1.0)
ax.set_title('Ablation jumlah fitur SFM'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_PNG,bbox_inches='tight'); plt.close()
meta=dict(deskripsi='Ablation jumlah fitur SFM (top-k dari importance model joint). MCC in-domain, joint, cross. Model A biner, z-score per dataset, XGBoost identik nb06, seed 42.',
          generated=datetime.datetime.utcnow().isoformat()+'Z', seed=SEED,
          feature_ranking=ranked, subsets=SUBSETS, results=rows)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('Saved:',OUT_JSON,'&',OUT_PNG)
try:
    import boto3; s3=boto3.client('s3')
    for fn,key in [(OUT_JSON,f'{S3_PREFIX}/ablation/feature_ablation.json'),(OUT_PNG,f'{S3_PREFIX}/ablation/feature_ablation.png')]:
        s3.upload_file(fn,S3_BUCKET,key); print('  upload ->',key)
except Exception as e:
    print('upload S3 dilewati:',e)
print('=== SEL 6 SELESAI ===')